# Start here: a small Instagram bar chart

This notebook is self-contained: it needs the graphics SDK but no prepared civic datasets. The values are **invented demonstration data**, not findings about Detroit. It shows the general pattern: a table → explicit encodings → a `Graphic` → preview and export.


In [ ]:
from pathlib import Path
import sys

# Run from this notebook folder or anywhere inside the Detroit checkout.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "strongtowns-data.lock.json").is_file()
             and (p / "projects/graphics/src/graphics").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open Jupyter inside the strongtowns-detroit checkout; see README.md.")
sys.path.insert(0, str(ROOT / "src"))
GRAPHICS = ROOT / "projects/graphics"
FORUM = ROOT / "projects/detroit-land-use-forum"
import strongtowns_graphics as graphics_sdk
if not hasattr(graphics_sdk, "GraphicInput"):
    raise RuntimeError(
        "This kernel has an older graphics SDK. Restart Jupyter with the uv command "
        "in README.md so it uses this project's locked dependencies."
    )
from IPython.display import SVG, display
from strongtowns_graphics import (
    GraphicFormat, GRAPHIC_FORMAT_SPECS, render_graphic_canvas,
    render_graphic_svg, write_graphic_bundle,
)

NOTEBOOK_NAME = "demo_bar_chart"
provenance = {"source": "Invented demonstration data; not Detroit findings"}


## A table you can replace

Each row names a category and two numeric quantities in the same unit. Replace this table with a CSV or your own analysis when you are ready.


In [ ]:
import polars as pl
from strongtowns_graphics import bar_chart, BarSeries, BarArrangement, BarOrientation, NumericAxis

data = pl.DataFrame({
    "category": ["Example A", "Example B", "Example C"],
    "existing": [12, 18, 9],
    "additional": [8, 4, 11],
})
display(data)


## Tell the renderer what each field means

The renderer does not decide your units, groups, or claim. Here we explicitly stack two series. Try `BarArrangement.GROUPED` to compare them side by side.


In [ ]:
graphic = bar_chart(
    data, category="category",
    arrangement=BarArrangement.STACKED,
    orientation=BarOrientation.VERTICAL,
    axis=NumericAxis(title="Example units", tick_step=5),
    series=(
        BarSeries(column="existing", label="Existing", color="#0c2340"),
        BarSeries(column="additional", label="Additional", color="#c8102e"),
    ),
    title="An example chart you can adapt",
    subtitle="Invented values for learning the graphics SDK",
    sources=("Source: demonstration data, not Detroit findings.",),
    description="A demonstration stacked chart: A has 12 existing and 8 additional; B has 18 and 4; C has 9 and 11.",
)
graphics = {"example-bar-chart": graphic}


## Preview at the standard publishing size


In [ ]:
# Change to GraphicFormat.INSTAGRAM_STORY for a story-sized export.
TARGET = GraphicFormat.INSTAGRAM_POST
EXPORT_PNG = True  # SVG and HTML work without the rsvg-convert system tool.
target = GRAPHIC_FORMAT_SPECS[TARGET]


In [ ]:
# Preview exactly the composition used by the export below.
for name, graphic in graphics.items():
    print(name)
    svg = (render_graphic_canvas(
        graphic, canvas_aspect_ratio=target.aspect_ratio,
        content_aspect_ratio=target.content_aspect_ratio,
        content_top_padding=target.content_top_padding,
    ) if target.content_aspect_ratio else render_graphic_svg(
        graphic, aspect_ratio=target.aspect_ratio,
    ))
    display(SVG(svg))
    print("Alt text:", graphic.description or graphic.title_text)


## Save the graphic

PNG requires the `rsvg-convert` system tool. See README.md or set `EXPORT_PNG = False` for SVG and HTML.


In [ ]:
import json
import shutil

output_dir = GRAPHICS / "output/notebooks" / NOTEBOOK_NAME / TARGET.value
formats = ("html", "svg", "png") if EXPORT_PNG else ("html", "svg")
if EXPORT_PNG and shutil.which("rsvg-convert") is None:
    raise RuntimeError(
        "PNG export needs rsvg-convert (see README.md). "
        "Set EXPORT_PNG = False above and rerun the export to save SVG/HTML now."
    )
for name, graphic in graphics.items():
    files = write_graphic_bundle(
        output_dir, name, graphic,
        aspect_ratio=target.aspect_ratio, png_width=target.png_width,
        content_aspect_ratio=target.content_aspect_ratio,
        content_top_padding=target.content_top_padding, formats=formats,
    )
    (output_dir / f"{name}.alt.txt").write_text(
        graphic.description or graphic.title_text, encoding="utf-8"
    )
    for kind, path in files.items():
        print(f"{kind}: {path}")
# Keep the exact source identities alongside your exported graphics.
(output_dir / "sources.json").write_text(
    json.dumps(provenance, indent=2) + "\n", encoding="utf-8"
)


## Next

Open [the parking chart](01_parking_gaps_by_project_type.ipynb) for the same pattern with actual prepared records, or [the BZA map](02_bza_cases_map.ipynb) for explicit point IDs, coordinates, categories, and marker magnitudes.
